In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
SEED = 0

np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)

N_STATES = 16
N_ACTIONS = 4

# Hole states + goal state
TERMINAL = {5, 7, 11, 12, 15}

print("FrozenLake environment created.")

FrozenLake environment created.


In [3]:
def one_hot(s):
    v = np.zeros(
        N_STATES,
        dtype=np.float32
    )

    v[s] = 1.0

    return v

In [4]:
def behavior_policy():

    # 30% random exploration
    if np.random.rand() < 0.3:
        return env.action_space.sample()

    # 70% choose only Down or Right
    return np.random.choice([1, 2])

In [5]:
S = []
A = []
R = []
S2 = []
D = []

for _ in range(300):

    s, _ = env.reset()
    done = False

    while not done:

        a = behavior_policy()

        s2, r, term, trunc, _ = env.step(a)

        done = term or trunc

        S.append(s)
        A.append(a)
        R.append(r)
        S2.append(s2)
        D.append(float(term))

        s = s2

# Convert lists to NumPy arrays
S = np.array(S)
A = np.array(A)
S2 = np.array(S2)

R = np.array(
    R,
    dtype=np.float32
)

D = np.array(
    D,
    dtype=np.float32
)

print("Number of transitions:", len(S))

Number of transitions: 1276


In [6]:
pair_counts = np.zeros(
    (N_STATES, N_ACTIONS),
    dtype=int
)

for s, a in zip(S, A):
    pair_counts[s, a] += 1

# Only non-terminal states
valid = [
    s
    for s in range(N_STATES)
    if s not in TERMINAL
]

# Common pairs: appeared at least 5 times
common = [
    (s, a)
    for s in valid
    for a in range(N_ACTIONS)
    if pair_counts[s, a] >= 5
]

# Rare pairs: appeared fewer than 5 times
rare = [
    (s, a)
    for s in valid
    for a in range(N_ACTIONS)
    if pair_counts[s, a] < 5
]

print("Common pairs:", len(common))
print("Rare pairs:", len(rare))

Common pairs: 38
Rare pairs: 6


In [7]:
class QNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(N_STATES, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, N_ACTIONS)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
def cql_loss(
    q_net,
    target_net,
    s,
    a,
    r,
    s2,
    done,
    gamma,
    alpha
):

    # Q-values for all actions
    q_all = q_net(s)

    # Q-value corresponding to dataset action
    q_data = q_all.gather(
        1,
        a.unsqueeze(1)
    ).squeeze(1)

    # Bellman target
    with torch.no_grad():

        target = (
            r
            + gamma
            * (1 - done)
            * target_net(s2).max(dim=1).values
        )

    bellman = F.mse_loss(
        q_data,
        target
    )

    # Conservative Q-Learning penalty
    cql_pen = (
        torch.logsumexp(
            q_all,
            dim=1
        )
        - q_data
    ).mean()

    return bellman + alpha * cql_pen

In [9]:
S_T = torch.tensor(
    np.array([
        one_hot(x)
        for x in S
    ])
)

S2_T = torch.tensor(
    np.array([
        one_hot(x)
        for x in S2
    ])
)

A_T = torch.tensor(
    A,
    dtype=torch.long
)

R_T = torch.tensor(R)

D_T = torch.tensor(D)

print("Dataset converted to tensors.")

Dataset converted to tensors.


In [10]:
def train(
    alpha,
    steps=3000,
    batch=64,
    lr=1e-3,
    gamma=0.99
):

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    q = QNet()
    tgt = QNet()

    tgt.load_state_dict(
        q.state_dict()
    )

    opt = torch.optim.Adam(
        q.parameters(),
        lr=lr
    )

    for step in range(steps):

        # Random mini-batch
        i = np.random.randint(
            0,
            len(S),
            batch
        )

        loss = cql_loss(
            q,
            tgt,
            S_T[i],
            A_T[i],
            R_T[i],
            S2_T[i],
            D_T[i],
            gamma,
            alpha
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        # Periodically update target network
        if step % 100 == 0:
            tgt.load_state_dict(
                q.state_dict()
            )

    return q

In [11]:
def q_table(q):

    states = torch.tensor(
        np.array([
            one_hot(s)
            for s in range(N_STATES)
        ])
    )

    with torch.no_grad():
        return q(states).numpy()

In [12]:
def evaluate(q):

    s, _ = env.reset()

    total = 0.0
    done = False
    steps = 0

    while not done and steps < 50:

        state_tensor = torch.tensor(
            one_hot(s)
        ).unsqueeze(0)

        with torch.no_grad():
            a = q(
                state_tensor
            ).argmax().item()

        s, r, term, trunc, _ = env.step(a)

        done = term or trunc

        total += r
        steps += 1

    return total

In [13]:
def avg_q(qt, pairs):

    return float(
        np.mean([
            qt[s, a]
            for s, a in pairs
        ])
    )

In [14]:
# alpha = 0 gives standard Offline DQN
dqn_model = train(alpha=0.0)

# alpha = 1 enables the CQL penalty
cql_model = train(alpha=1.0)

dqn_q = q_table(dqn_model)
cql_q = q_table(cql_model)

dqn_ret = evaluate(dqn_model)
cql_ret = evaluate(cql_model)

print("Training completed.")

Training completed.


In [15]:
print(
    f"Dataset: {len(S)} transitions | "
    f"Left used {100*np.mean(A == 0):.0f}%, "
    f"Down {100*np.mean(A == 1):.0f}%, "
    f"Right {100*np.mean(A == 2):.0f}%, "
    f"Up {100*np.mean(A == 3):.0f}%"
)

print(
    f"Rare/unseen (state,action) pairs: "
    f"{len(rare)} of {len(rare) + len(common)}\n"
)

print(
    f"{'':<12}"
    f"{'Return':>8}"
    f"{'Avg Q common':>15}"
    f"{'Avg Q rare':>13}"
    f"{'Rare Q > 1.0':>15}"
)

for name, qt, ret in [
    ("Offline DQN", dqn_q, dqn_ret),
    ("CQL", cql_q, cql_ret)
]:

    over = sum(
        qt[s, a] > 1.0
        for s, a in rare
    )

    print(
        f"{name:<12}"
        f"{ret:>8.1f}"
        f"{avg_q(qt, common):>15.3f}"
        f"{avg_q(qt, rare):>13.3f}"
        f"{over:>11d}/{len(rare)}"
    )

print(
    "\nNote: the max possible reward is 1.0, "
    "so any Q above 1.0 is overestimation."
)

Dataset: 1276 transitions | Left used 7%, Down 42%, Right 43%, Up 7%
Rare/unseen (state,action) pairs: 6 of 44

              Return   Avg Q common   Avg Q rare   Rare Q > 1.0
Offline DQN      0.0          0.808        0.660          0/6
CQL              0.0          1.245        0.488          1/6

Note: the max possible reward is 1.0, so any Q above 1.0 is overestimation.
